# Apply more and more complex models to the data

In [10]:
import numpy as np
from pathlib import Path
import os
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import warnings
from sklearn.model_selection import KFold
import os, json
import pandas as pd

# Ensure working directory points to LEAD_ExperimentalFolder so module LEAD is found
desired_cwd = Path('/home/thardy/elefanto/ConsciousnessTeam_Data/SOUNDMODEL/Data_SoundGOOD/LEAD_ExperimentalFolder')
if Path.cwd() != desired_cwd:
    os.chdir(desired_cwd)

import LEAD as lead


# Important variables
cwd = Path.cwd()
SubIDs = ['01','02','03','05','06','07','08','09','11','12','13','14','15','17','19','20','22','23','24','25']
colormap = {0: (0, 0, 0), 1: (0, 0.25, 1), 2: (0, 0.9375, 1), 3: (0, 0.91, 0.1), 4: (1, 0.6, 0), 5: (1, 0, 0), 6: (0.8, 0, 0)}

In [11]:
checkpoint_path="Application_EEG_v2_Active_late.json"
with open(checkpoint_path, "r") as f:
    checkpoint = json.load(f)

# Save LEAD activity around peak for each fold of the 5CV

In [ ]:
def generate_distributions(state_series, input_series, prefitted_linear_params, prefitted_nonlinear_params, prefitted_gainmodul_params):
    """Generate model distributions using prefitted parameters."""

    n_categories = len(list(state_series.keys()))
    categories = list(range(n_categories))
    
    kf = KFold(n_splits=5, shuffle=True, random_state=0)
    trial_indices = np.arange(np.min([state_series[cat].shape[0] for cat in categories]))

    fold_distributions = {}

    for foldidx, (train_idx, test_idx) in enumerate(kf.split(trial_indices)):
        # Split data
        state_train, input_train = {}, {}
        state_test, input_test = {}, {}
        for cat in categories:
            state_train[cat] = state_series[cat][train_idx]
            input_train[cat] = input_series[cat][train_idx]
            state_test[cat] = state_series[cat][test_idx]
            input_test[cat] = input_series[cat][test_idx]

        # ---- Create models instances ----
        linear = lead.model.StratifiedLinear(tau=10, process_noise=0.1, measure_noise=0.1, w0=0)
        linear.set_params(prefitted_linear_params[foldidx])
        gainmodul = lead.model.StratifiedGainModulation(tau=10, process_noise=0.1, measure_noise=0.1, threshold=1, sharpness=5)
        gainmodul.set_params(prefitted_gainmodul_params[foldidx])
        nonlinear = lead.model.StratifiedNonLinear1(tau=10, process_noise=0.1, measure_noise=0.1, gain=0.1, threshold=1, sharpness=5)
        nonlinear.set_params(prefitted_nonlinear_params[foldidx])

        # ----  Run samples ----
        n_trials = 1000
        one_input = np.concatenate((np.zeros(75), np.ones(25)))
        sim_input_series = {cat: np.stack([one_input for _ in range(n_trials)]) for cat in categories}
        measures_linear = linear.measure_simulations(input_series = sim_input_series)
        measures_nonlinear = nonlinear.measure_simulations(input_series = sim_input_series)
        measures_gainmodul = gainmodul.measure_simulations(input_series = sim_input_series)

        # Build per-category distributions (for 6x rows later)
        true_distribs_by_cat = {cat: np.mean(state_test[cat][:, 90:100], 1) for cat in categories}
        linear_distribs_by_cat = {cat: np.mean(measures_linear[cat][:, 90:100], 1) for cat in categories}
        nonlinear_distribs_by_cat = {cat: np.mean(measures_nonlinear[cat][:, 90:100], 1) for cat in categories}
        gainmodul_distribs_by_cat = {cat: np.mean(measures_gainmodul[cat][:, 90:100], 1) for cat in categories}


        # Store the distributions for this fold (keeping snr info per point and per category)
        fold_distributions[foldidx] = {
            'real': true_distribs_by_cat,
            'linear': linear_distribs_by_cat,
            'nonlinear': nonlinear_distribs_by_cat,
            'gainmodul': gainmodul_distribs_by_cat,
        }

    return fold_distributions

In [16]:
version = 3

task = 'Active'
def run_summarization(save=True):

    # Load previous results
    old_checkpoint_path = f"Application_EEG_v3_{task}_late.json"
    with open(old_checkpoint_path, "r") as f:
        old_checkpoint = json.load(f)
    
    ll_list_all_parts = old_checkpoint['detailed']
    linear_params_list_all_parts, nonlinear_params_list_all_parts, gainmodul_params_list_all_parts = [], [], []
    
    for part in range(20):
        linear_params_list_this_part, nonlinear_params_list_this_part, gainmodul_params_list_this_part = [], [], []
        for foldidx in range(5):
            linear_params_list_this_part.append(old_checkpoint['fitted_params'][part][foldidx][0])
            nonlinear_params_list_this_part.append(old_checkpoint['fitted_params'][part][foldidx][1])
            gainmodul_params_list_this_part.append(old_checkpoint['fitted_params'][part][foldidx][2])
        linear_params_list_all_parts.append(linear_params_list_this_part)
        nonlinear_params_list_all_parts.append(nonlinear_params_list_this_part)
        gainmodul_params_list_all_parts.append(gainmodul_params_list_this_part)
    
    # List to collect rows for the CSV
    distribution_records = []

    for partidx, part in enumerate(range(20)):
        print(f"Processing Participant: {part}")
        
        # ... (Data loading and input definition remains the same)
        data_ref = f'myEpochs_{task}/Epoch_{SubIDs[part]}-epo.fif'
        epochs_file = cwd.parents[0] / data_ref
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            state_series = lead.STG(epochs_file, tmin=300, tmax=500)
            n_categories = len(list(state_series.keys()))
            categories = list(range(n_categories))

        one_input = np.concatenate((np.zeros(75), np.ones(25), np.zeros(150)))
        input_series = {cat: np.stack([one_input for _ in range(state_series[cat].shape[0])]) for cat in categories}
        
        fold_dist = generate_distributions(state_series, input_series, linear_params_list_all_parts[partidx], 
                                      nonlinear_params_list_all_parts[partidx], 
                                      gainmodul_params_list_all_parts[partidx])
        
        # Process fold_dist into a flat format for CSV
        # Desired: one row per (model or true) x fold x cat (snr), with snr flagged explicitly.
        for foldidx, models in fold_dist.items():
            for model_name, values in models.items():
                for cat, array_values in values.items():
                    record = {
                        "participant": part,
                        "fold": foldidx,
                        "model": model_name,
                        "snr": cat 
                    }
                    for i, val in enumerate(array_values):
                        record[f"idx_{i}"] = val
                    distribution_records.append(record)

        # ---- Save distributions to CSV ----
        if save:
            csv_path = f"Distributions_v{version}_{task}_late_full.csv"
            df_dist = pd.DataFrame(distribution_records)
            df_dist.to_csv(csv_path, index=False)

    return distribution_records


distribution_records = run_summarization(save=False)

Processing Participant: 0
Processing Participant: 1
Processing Participant: 2
Processing Participant: 3
Processing Participant: 4
Processing Participant: 5
Processing Participant: 6
Processing Participant: 7
Processing Participant: 8
Processing Participant: 9
Processing Participant: 10
Processing Participant: 11
Processing Participant: 12
Processing Participant: 13
Processing Participant: 14
Processing Participant: 15
Processing Participant: 16
Processing Participant: 17
Processing Participant: 18
Processing Participant: 19
